SECTION 1

In [1]:
!nvidia-smi

Thu Oct  8 07:06:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
from numba import cuda
import os

device = cuda.get_current_device()

# Fixed: Removed .decode('utf-8') since device.name is already a string
print(f"Device Name: {device.name}")
print(f"Compute Capability: {device.compute_capability}")
print(f"Multiprocessor Count (SMs): {getattr(device, 'MULTIPROCESSOR_COUNT', 'N/A')}")
print(f"Max Threads Per Block: {device.MAX_THREADS_PER_BLOCK}")
print(f"Max Block Dimensions: {device.MAX_BLOCK_DIM_X}, {device.MAX_BLOCK_DIM_Y}, {device.MAX_BLOCK_DIM_Z}")
print(f"Warp Size: {device.WARP_SIZE}")

Device Name: Tesla T4
Compute Capability: ComputeCapability(major=7, minor=5)
Multiprocessor Count (SMs): 40
Max Threads Per Block: 1024
Max Block Dimensions: 1024, 1024, 64
Warp Size: 32


SECTION 2

In [4]:
import numpy as np
import time
from numba import cuda

# 1. CPU Reference Implementation (Pure Python loop)
def double_cpu_pure(arr):
    out = np.empty_like(arr)
    for i in range(len(arr)):
        out[i] = arr[i] * 2.0
    return out

# 2. GPU Kernel Implementation
@cuda.jit
def double_gpu_kernel(d_in, d_out):
    # Calculate unique 1D thread ID
    idx = cuda.grid(1)
    if idx < d_in.shape[0]:
        d_out[idx] = d_in[idx] * 2.0

# 3. Benchmark Harness across array sizes
def benchmark_run(N, student_id_seed=1234):
    np.random.seed(student_id_seed)
    h_in = np.random.rand(N).astype(np.float32)

    # Measure CPU Pure Loop (skip for N >= 10M if too slow)
    if N <= 1_000_000:
        t0 = time.perf_counter()
        _ = double_cpu_pure(h_in)
        cpu_time_ms = (time.perf_counter() - t0) * 1000.0
    else:
        cpu_time_ms = float('nan') # Skip/Extrapolate

    # GPU Setup
    threads_per_block = 256
    blocks_per_grid = (N + threads_per_block - 1) // threads_per_block

    # Warmup compilation run
    d_warmup = cuda.to_device(np.ones(10, dtype=np.float32))
    double_gpu_kernel[1, 32](d_warmup, d_warmup)
    cuda.synchronize()

    # GPU Roundtrip Measurement (Includes PCIe transfers)
    t0 = time.perf_counter()
    d_in = cuda.to_device(h_in)
    d_out = cuda.device_array_like(d_in)
    double_gpu_kernel[blocks_per_grid, threads_per_block](d_in, d_out)
    h_out = d_out.copy_to_host()
    cuda.synchronize()
    gpu_total_ms = (time.perf_counter() - t0) * 1000.0

    # GPU Compute-Only Measurement (Pre-allocated in VRAM)
    t0 = time.perf_counter()
    double_gpu_kernel[blocks_per_grid, threads_per_block](d_in, d_out)
    cuda.synchronize()
    gpu_compute_ms = (time.perf_counter() - t0) * 1000.0

    return cpu_time_ms, gpu_total_ms, gpu_compute_ms

# Run benchmark suite across specified sample sizes
sizes = [100, 1_000, 10_000, 100_000, 1_000_000, 5_000_000, 10_000_000]

print(f"{'Array Size (N)':<15} | {'CPU Time (ms)':<15} | {'GPU Total (ms)':<15} | {'GPU Kernel Only (ms)':<20} | {'Winner':<10}")
print("-" * 85)

for N in sizes:
    cpu_ms, gpu_tot_ms, gpu_kern_ms = benchmark_run(N)

    if np.isnan(cpu_ms):
        winner = "GPU"
        cpu_str = "N/A"
    else:
        winner = "CPU" if cpu_ms < gpu_tot_ms else "GPU"
        cpu_str = f"{cpu_ms:.4f}"

    print(f"{N:<15,} | {cpu_str:<15} | {gpu_tot_ms:<15.4f} | {gpu_kern_ms:<20.4f} | {winner:<10}")

Array Size (N)  | CPU Time (ms)   | GPU Total (ms)  | GPU Kernel Only (ms) | Winner    
-------------------------------------------------------------------------------------


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 40 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


100             | 0.0694          | 1.5649          | 0.1537               | CPU       
1,000           | 0.5173          | 0.7749          | 0.0754               | CPU       
10,000          | 2.6727          | 0.4385          | 0.0588               | GPU       
100,000         | 28.9972         | 0.6084          | 0.0642               | GPU       
1,000,000       | 237.6235        | 4.6488          | 0.1199               | GPU       
5,000,000       | N/A             | 13.1918         | 0.2560               | GPU       
10,000,000      | N/A             | 24.5358         | 0.4483               | GPU       


Q1. What is the empirical Crossover Point N* on your machine? (i.e., at what array size did
GPU Total Time finally become faster than CPU Time?):

A: N* = 10,000(This is the first array size where GPU Total Time of 0.4385 ms became faster than CPU Time of 2.6727 ms).

Q2. At N = 100, why is GPU Total Time dramatically slower than CPU Time? Compare the
'GPU Total' column against the 'GPU Kernel Only' column to explain where 95%+ of the
elapsed time was actually spent:

A: At N = 100, the pure computation on the GPU takes only 0.1537 ms (GPU Kernel Only), whereas total execution takes 1.5649 ms.
Over 90%-95%+ of the elapsed time is consumed by PCIe bus overhead—specifically host-to-device memory allocation,
data transfers across the physical PCIe bus, driver setup, and context synchronization-making CPU processing in local cache vastly superior for tiny datasets.

SECTION 3

In [7]:
# DEFECTIVE KERNEL: Notice the missing 'if idx < n' check!
@cuda.jit
def defective_kernel(d_arr):
    idx = cuda.grid(1)
    # FORBIDDEN: Writing without bounds check!
    d_arr[idx] = 999.0

# TEST HARNESS
N = 1000
h_arr = np.zeros(N, dtype=np.float32)
d_arr = cuda.to_device(h_arr)

# Launch 4 blocks of 256 = 1024 threads
threads_per_block = 256
blocks_per_grid = 4
print("Launching defective kernel with 1024 threads on 1000-element array...")
defective_kernel[blocks_per_grid, threads_per_block](d_arr)
cuda.synchronize()
print("Execution finished (or did it crash?). Checking host copy...")
h_result = d_arr.copy_to_host()
print(f"Elements processed: {len(h_result)}. Last element: {h_result[-1]}")

Launching defective kernel with 1024 threads on 1000-element array...
Execution finished (or did it crash?). Checking host copy...
Elements processed: 1000. Last element: 999.0


In [8]:
import numpy as np
from numba import cuda

@cuda.jit
def inspect_threads_kernel(log_block, log_thread, log_global, log_active, N):
    t_idx = cuda.threadIdx.x
    b_idx = cuda.blockIdx.x
    g_idx = cuda.grid(1)

    if g_idx < log_block.shape[0]:
        log_block[g_idx] = b_idx
        log_thread[g_idx] = t_idx
        log_global[g_idx] = g_idx
        log_active[g_idx] = 1 if g_idx < N else 0

# Configuration
N = 37
threads_per_block = 8
blocks_per_grid = (N + threads_per_block - 1) // threads_per_block # Exactly 5 blocks = 40 threads

log_b = cuda.device_array(blocks_per_grid * threads_per_block, dtype=np.int32)
log_t = cuda.device_array(blocks_per_grid * threads_per_block, dtype=np.int32)
log_g = cuda.device_array(blocks_per_grid * threads_per_block, dtype=np.int32)
log_a = cuda.device_array(blocks_per_grid * threads_per_block, dtype=np.int32)

inspect_threads_kernel[blocks_per_grid, threads_per_block](log_b, log_t, log_g, log_a, N)
cuda.synchronize()

print(f"{'Global ID':<10} | {'Block ID':<10} | {'Thread ID':<10} | {'Status':<10}")
print("-" * 50)
for i in range(32, 40): # Print the critical boundary transition (Block 4)
    status = "ACTIVE" if log_a[i] == 1 else "IDLE (GUARDED)"
    print(f"{log_g[i]:<10} | {log_b[i]:<10} | {log_t[i]:<10} | {status:<10}")

Global ID  | Block ID   | Thread ID  | Status    
--------------------------------------------------
32         | 4          | 0          | ACTIVE    
33         | 4          | 1          | ACTIVE    
34         | 4          | 2          | ACTIVE    
35         | 4          | 3          | ACTIVE    
36         | 4          | 4          | ACTIVE    
37         | 4          | 5          | IDLE (GUARDED)
38         | 4          | 6          | IDLE (GUARDED)
39         | 4          | 7          | IDLE (GUARDED)


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 5 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


Total Blocks Allocated: 5

Total Hardware Threads Launched: 40

Number of Active Working Threads: 37

Number of Idle Guarded Threads: 3

SECTION 4

In [9]:
import numpy as np
import time
from numba import cuda

# BUGGY NAIVE SUMMATION
@cuda.jit
def naive_sum_kernel(d_arr, d_total):
    idx = cuda.grid(1)
    if idx < d_arr.shape[0]:
        # DATA RACE: Thousands of threads execute Read-Modify-Write at the exact same time!
        d_total[0] += d_arr[idx]

# CORRECT ATOMIC SUMMATION
@cuda.jit
def atomic_sum_kernel(d_arr, d_total):
    idx = cuda.grid(1)
    if idx < d_arr.shape[0]:
        # HARDWARE ATOMIC LOCK: Hardware serializes updates to prevent collisions
        cuda.atomic.add(d_total, 0, d_arr[idx])

# Experiment Setup
N = 50_000
threads_per_block = 256
blocks_per_grid = (N + threads_per_block - 1) // threads_per_block

# Host & Device Memory Setup
h_arr = np.ones(N, dtype=np.float32)
d_arr = cuda.to_device(h_arr)

# Warmup compilation
d_dummy_tot = cuda.to_device(np.zeros(1, dtype=np.float32))
naive_sum_kernel[blocks_per_grid, threads_per_block](d_arr, d_dummy_tot)
atomic_sum_kernel[blocks_per_grid, threads_per_block](d_arr, d_dummy_tot)
cuda.synchronize()

print("--- TRIAL 1 to 5: NAIVE REDUCTION (RACE CONDITION) ---")
naive_times = []
for i in range(5):
    d_total = cuda.to_device(np.zeros(1, dtype=np.float32))

    t0 = time.perf_counter()
    naive_sum_kernel[blocks_per_grid, threads_per_block](d_arr, d_total)
    cuda.synchronize()
    elapsed_ms = (time.perf_counter() - t0) * 1000.0
    naive_times.append(elapsed_ms)

    output = d_total.copy_to_host()[0]
    dev = 50000.0 - output
    print(f"Trial {i+1}: Target = 50,000.0 | Output = {output:<8.1f} | Deviation = {dev:<8.1f}")

print("\n--- ATOMIC REDUCTION ---")
d_total_atomic = cuda.to_device(np.zeros(1, dtype=np.float32))
t0 = time.perf_counter()
atomic_sum_kernel[blocks_per_grid, threads_per_block](d_arr, d_total_atomic)
cuda.synchronize()
atomic_time_ms = (time.perf_counter() - t0) * 1000.0
atomic_output = d_total_atomic.copy_to_host()[0]

print(f"Atomic Output: {atomic_output:.1f}")
print(f"Naive Avg Time: {np.mean(naive_times):.4f} ms | Atomic Time: {atomic_time_ms:.4f} ms")

--- TRIAL 1 to 5: NAIVE REDUCTION (RACE CONDITION) ---
Trial 1: Target = 50,000.0 | Output = 2.0      | Deviation = 49998.0 
Trial 2: Target = 50,000.0 | Output = 2.0      | Deviation = 49998.0 
Trial 3: Target = 50,000.0 | Output = 2.0      | Deviation = 49998.0 
Trial 4: Target = 50,000.0 | Output = 2.0      | Deviation = 49998.0 
Trial 5: Target = 50,000.0 | Output = 2.0      | Deviation = 49998.0 

--- ATOMIC REDUCTION ---
Atomic Output: 50000.0
Naive Avg Time: 0.1358 ms | Atomic Time: 0.4156 ms


Atomic Sum Output: 50000.0 (Yes, it equals exactly 50,000.0)

Naive Kernel Time: 0.1358 ms vs. Atomic Kernel Time: 0.4156 ms

Explain why atomic operations incur a massive execution penalty:

Atomic operations enforce a hardware-level lock on the target memory location (d_total[0]). Instead of allowing thousands of GPU threads to execute writes in parallel across all cores simultaneously, atomic operations force the hardware memory controller to serialize incoming requests. Each thread must wait its turn in a sequential queue to complete its Read-Modify-Write operation, leading to severe execution stalls and significant memory latency overhead.

SECTION 5

In [10]:
import numpy as np
import math
from numba import cuda

@cuda.jit
def radial_vignette_2d(d_canvas, width, height, center_x, center_y):
    # Retrieve 2D spatial coordinates directly from the grid
    x, y = cuda.grid(2)
    if x < width and y < height:
        dx = float(x - center_x)
        dy = float(y - center_y)
        dist = math.sqrt(dx * dx + dy * dy)
        max_dist = math.sqrt(float(center_x**2 + center_y**2))
        intensity = 1.0 - (dist / max_dist)
        if intensity < 0.0:
            intensity = 0.0
        d_canvas[y, x] = intensity

# 2D EXECUTION CONFIGURATION
WIDTH, HEIGHT = 1024, 1024
canvas = np.zeros((HEIGHT, WIDTH), dtype=np.float32)
d_canvas = cuda.to_device(canvas)

# 2D Blocks (16x16 = 256 threads per block)
threads_2d = (16, 16)
blocks_2d = (
    (WIDTH + threads_2d[0] - 1) // threads_2d[0],
    (HEIGHT + threads_2d[1] - 1) // threads_2d[1]
)

radial_vignette_2d[blocks_2d, threads_2d](d_canvas, WIDTH, HEIGHT, WIDTH // 2, HEIGHT // 2)
cuda.synchronize()

result_canvas = d_canvas.copy_to_host()

print(f"Grid Dimensions: Blocks in X: {blocks_2d[0]} | Blocks in Y: {blocks_2d[1]} | Total Blocks: {blocks_2d[0] * blocks_2d[1]}")
print(f"Verified Pixel Value at Center Point (512, 512): {result_canvas[512, 512]:.4f}")
print(f"Verified Pixel Value at Outer Corner Point (0, 0): {result_canvas[0, 0]:.4f}")

Grid Dimensions: Blocks in X: 64 | Blocks in Y: 64 | Total Blocks: 4096
Verified Pixel Value at Center Point (512, 512): 1.0000
Verified Pixel Value at Outer Corner Point (0, 0): 0.0000


SECTION 6

In [12]:
import hashlib

def generate_lab_checksum(student_id_str, d_canvas, exp_table_crossover):
    h = hashlib.sha256()
    h.update(student_id_str.strip().encode('utf-8'))

    # Encode GPU name string to bytes
    gpu_name = str(cuda.get_current_device().name)
    h.update(gpu_name.encode('utf-8'))

    # Include sample of compute output
    h.update(d_canvas.copy_to_host()[:10, :10].tobytes())
    h.update(str(exp_table_crossover).encode('utf-8'))

    checksum_hash = h.hexdigest()[:16].upper()
    return checksum_hash

STUDENT_ID = "230103317"
crossover_point_n = 10000

token = generate_lab_checksum(STUDENT_ID, d_canvas, crossover_point_n)
print("=" * 60)
print(f"FINAL LAB CHECKSUM TOKEN: {token}")
print("=" * 60)

FINAL LAB CHECKSUM TOKEN: 29DB34B2803439CE


TOKEN: 29DB34B2803439CE
